In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from moc.configs.config import get_config
from moc.utils.run_config import RunConfig
# from moc.models.mqf2.lightning_module import MQF2LightningModule
from moc.models.mixture.mixture_model2 import MixtureLightningModule
from moc.models.gaussian.gaussian import GaussianLightningModule
from moc.models.trainers.lightning_trainer import get_lightning_trainer
from moc.datamodules.generated import GeneratedDataModule
import numpy as np
import matplotlib.pyplot as plt
plt.style.use('seaborn-v0_8')
from moc.metrics.distribution_metrics import pce
import torch
from torch.distributions import MultivariateNormal

In [3]:
torch.manual_seed(42)

In [4]:
def build_multivariate_normal(d, sigma2, tau, mean):
    indices = torch.arange(d).unsqueeze(0)
    covariance_matrix = sigma2 * torch.exp(-torch.abs(indices.T - indices) / tau)
    return MultivariateNormal(mean, covariance_matrix)

In [5]:
d = 10
sigma2 = 1.0
tau = 1.0

mean = torch.zeros(d)
true_dist = build_multivariate_normal(d, sigma2, tau, mean)
# undercorr = build_multivariate_normal(d, sigma2, tau, mean)

In [6]:
undermean = torch.ones(d) * - 0.5
overmean = torch.ones(d) * 0.5
undermean_dist = build_multivariate_normal(d, sigma2, tau, undermean)
overmean_dist = build_multivariate_normal(d, sigma2, tau, overmean)
undervar_dist = build_multivariate_normal(d, 0.85, tau, mean)
overvar_dist = build_multivariate_normal(d, 1.25, tau, mean)
undercorr_dist = build_multivariate_normal(d, sigma2, 0.5, mean)
overcorr_dist = build_multivariate_normal(d, sigma2, 2, mean)


In [7]:
N = 10000
y = true_dist.sample((N,))
y.shape

torch.Size([10000, 10])

In [18]:
pces, var = pce(overcorr_dist, y, n_samples = 20, mode = 'all', prerank = 'density', setup='simulated')

pit values have shape torch.Size([1, 10000])


In [ ]:
np.sum(pces*var)

[0.2013929784297943]

In [5]:
config = get_config()
config.device = 'cuda'
csv_path = 'generated_datasets/linear_depX-False_coup-False_nS-3071_nF-14_nT-8_id-8.csv'

In [7]:
seeds = [42]
prerank = 'marginal'
rc = RunConfig(config, 'generated', "")
for seed in seeds:
    datamodule = GeneratedDataModule(csv_path=csv_path, seed=seed, num_workers=8)
    p, q = datamodule.input_dim, datamodule.output_dim
    model = GaussianLightningModule(p, q, prerank = prerank)
    # model = MixtureLightningModule(p, q, prerank = prerank)
    #model = MQF2LightningModule(p, q)
    trainer = get_lightning_trainer(rc)
    trainer.fit(model, datamodule)
    # wandb.finish()
    model.to(config.device)
    # pce_over_seeds = np.array(pce_over_seeds)

AttributeError: 'NoneType' object has no attribute 'dataset_group'